# Right-Side Common-Support Matching Readiness

The first right-side error-specific intervention produced a strong late-state signal, but the both-correct and right-error cohorts differed substantially in their physical configurations.

[实验设置与运行说明](../../../docs/experiment_notes/11_handoff_development_and_matching/fd8c801390fc1219.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 10 tracked cells.')


## 1. Frozen behavior cohorts and physical support


In [ ]:
%%cell_progress 1/10 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/10 Verify the analysis environment
cell_stage('Loading the CPU-only matching stack')
import json
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import csr_matrix
print('numpy / pandas / scipy:', np.__version__, pd.__version__, scipy.__version__)
assert hasattr(scipy.optimize, 'milp'), 'SciPy milp is required.'


In [ ]:
%%cell_progress 3/10 Lock common-support matching rules
cell_stage('Freezing features, calipers, balance gates, and optimization order')
import hashlib
from datetime import datetime, timezone

SEED = 20261118
MINIMUM_MATCHED_UNITS = 8
TARGET_MATCHED_UNITS = 10
MAXIMUM_ABSOLUTE_SMD = 0.50
MAXIMUM_MARGIN_BIN_DIFFERENCE = 1
MAXIMUM_STANDARDIZED_PAIR_DISTANCE = 3.00
BALANCE_BOUNDS = (0.50, 0.40, 0.30, 0.20, 0.15, 0.10, 0.05)
REQUIRED_PATTERNS = (
    'left_correct__right_correct',
    'left_correct__right_error',
)
PHYSICAL_FEATURES = (
    'units_1', 'distance_1', 'units_2', 'distance_2',
    'total_units', 'unit_difference',
    'total_distance', 'distance_difference',
    'smaller_side_torque', 'larger_side_torque',
    'absolute_net_torque',
)

READINESS_ROOT = (
    PROJECT_ROOT / 'results' / 'side_correctness_2x2_readiness_v3'
)
READINESS_RUN_PATH = READINESS_ROOT / 'RUN_COMPLETE.json'
READINESS_PROTOCOL_PATH = READINESS_ROOT / 'frozen_protocol.json'
READINESS_CANDIDATE_PATH = READINESS_ROOT / 'paired_candidates.jsonl'
READINESS_PAIR_TABLE_PATH = READINESS_ROOT / 'pair_readiness.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'rightside_common_support_matching_readiness_v1'
)
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'frozen_protocol.json'
FEATURE_TABLE_PATH = OUTPUT_ROOT / 'pair_physical_features.csv'
EDGE_TABLE_PATH = OUTPUT_ROOT / 'admissible_matching_edges.csv'
SEARCH_PATH = OUTPUT_ROOT / 'optimization_search.csv'
SELECTED_MATCH_PATH = OUTPUT_ROOT / 'selected_common_support_matches.jsonl'
SELECTED_SCENE_PATH = OUTPUT_ROOT / 'selected_common_support_scenes.jsonl'
BALANCE_PATH = OUTPUT_ROOT / 'selected_balance.csv'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for path in (OUTPUT_ROOT, ANALYSIS_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value!r}'.encode('utf-8')).hexdigest()

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)

def write_json_atomic(path, value):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    temporary.write_text(json.dumps(
        value, indent=2, ensure_ascii=False, default=json_default
    ) + '\n', encoding='utf-8')
    temporary.replace(path)

def write_jsonl_atomic(path, rows):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(
                row, sort_keys=True, default=json_default
            ) + '\n')
    temporary.replace(path)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

for required in (
    READINESS_RUN_PATH, READINESS_PROTOCOL_PATH,
    READINESS_CANDIDATE_PATH, READINESS_PAIR_TABLE_PATH,
):
    assert Path(required).is_file(), f'Missing prerequisite: {required}'
readiness_run = json.loads(READINESS_RUN_PATH.read_text(encoding='utf-8'))
assert readiness_run['primary_decision'] == 'side_by_correctness_2x2_support_insufficient'

protocol_identity = {
    'schema_version': 'rightside-common-support-matching-readiness-1.0',
    'analysis_status': 'behavior-and-physics-only pre-causal matching audit',
    'readiness_run_sha256': sha256_file(READINESS_RUN_PATH),
    'readiness_candidate_sha256': sha256_file(READINESS_CANDIDATE_PATH),
    'readiness_pair_table_sha256': sha256_file(READINESS_PAIR_TABLE_PATH),
    'cohort_patterns': list(REQUIRED_PATTERNS),
    'physical_features': list(PHYSICAL_FEATURES),
    'exact_match': 'diagnostic_cell',
    'maximum_margin_bin_difference': MAXIMUM_MARGIN_BIN_DIFFERENCE,
    'maximum_standardized_pair_distance': MAXIMUM_STANDARDIZED_PAIR_DISTANCE,
    'balance_bounds': list(BALANCE_BOUNDS),
    'post_match_maximum_absolute_smd': MAXIMUM_ABSOLUTE_SMD,
    'minimum_matched_units': MINIMUM_MATCHED_UNITS,
    'target_matched_units': TARGET_MATCHED_UNITS,
    'uses_hidden_states': False,
    'uses_intervention_results': False,
    'seed': SEED,
}
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Existing protocol differs. Use a new versioned output directory.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
print('Minimum / preferred matched units:',
      MINIMUM_MATCHED_UNITS, TARGET_MATCHED_UNITS)
print('Maximum post-match |SMD|:', MAXIMUM_ABSOLUTE_SMD)


In [ ]:
%%cell_progress 4/10 Reconstruct orientation-neutral physical features
cell_stage('Loading all correct and right-error mirror pairs')
from collections import Counter, defaultdict

candidate_manifest = read_jsonl(READINESS_CANDIDATE_PATH)
candidate_by_pair = defaultdict(list)
for row in candidate_manifest:
    candidate_by_pair[str(row['pair_id'])].append(row)
pair_frame = pd.read_csv(READINESS_PAIR_TABLE_PATH)
pair_frame['pair_id'] = pair_frame['pair_id'].astype(str)
pair_frame = pair_frame[pair_frame['pattern'].isin(REQUIRED_PATTERNS)].copy()
assert pair_frame['pattern'].value_counts().to_dict() == {
    'left_correct__right_error': 69,
    'left_correct__right_correct': 14,
}

def physical_record(pair_row):
    pair_id = str(pair_row['pair_id'])
    rows = candidate_by_pair[pair_id]
    anchor = next(row for row in rows if row['orientation'] == 'anchor')
    geometry = sorted(tuple(map(int, item)) for item in anchor['pair_geometry'])
    (units_1, distance_1), (units_2, distance_2) = geometry
    torques = sorted((
        int(anchor['left_torque']), int(anchor['right_torque'])
    ))
    return {
        'pair_id': pair_id,
        'pattern': str(pair_row['pattern']),
        'cohort': (
            'both_correct' if pair_row['pattern'] == REQUIRED_PATTERNS[0]
            else 'right_error'
        ),
        'diagnostic_cell': str(anchor['diagnostic_cell']),
        'margin_bin': int(anchor['margin_bin']),
        'units_1': units_1, 'distance_1': distance_1,
        'units_2': units_2, 'distance_2': distance_2,
        'total_units': int(anchor['left_units']) + int(anchor['right_units']),
        'unit_difference': abs(
            int(anchor['left_units']) - int(anchor['right_units'])
        ),
        'total_distance': (
            int(anchor['left_distance']) + int(anchor['right_distance'])
        ),
        'distance_difference': abs(
            int(anchor['left_distance']) - int(anchor['right_distance'])
        ),
        'smaller_side_torque': torques[0],
        'larger_side_torque': torques[1],
        'absolute_net_torque': abs(int(anchor['net_torque'])),
    }

feature_frame = pd.DataFrame([
    physical_record(row) for _, row in pair_frame.iterrows()
])
feature_frame.to_csv(FEATURE_TABLE_PATH, index=False)
assert feature_frame['pair_id'].nunique() == len(feature_frame) == 83
print('Cohort sizes:', feature_frame['cohort'].value_counts().to_dict())
display(feature_frame.groupby('cohort')[list(PHYSICAL_FEATURES)].mean().round(3))


In [ ]:
%%cell_progress 5/10 Build exact-category and calipered matching edges
cell_stage('Restricting the comparison to observable common support')

correct_frame = feature_frame[feature_frame['cohort'].eq('both_correct')].reset_index(drop=True)
error_frame = feature_frame[feature_frame['cohort'].eq('right_error')].reset_index(drop=True)
pooled_scale = feature_frame[list(PHYSICAL_FEATURES)].std(ddof=0).replace(0, 1.0)
pooled_mean = feature_frame[list(PHYSICAL_FEATURES)].mean()
correct_z = (
    correct_frame[list(PHYSICAL_FEATURES)] - pooled_mean
) / pooled_scale
error_z = (
    error_frame[list(PHYSICAL_FEATURES)] - pooled_mean
) / pooled_scale

edge_rows = []
for correct_index, correct in correct_frame.iterrows():
    for error_index, error in error_frame.iterrows():
        if correct['diagnostic_cell'] != error['diagnostic_cell']:
            continue
        margin_difference = abs(int(correct['margin_bin']) - int(error['margin_bin']))
        if margin_difference > MAXIMUM_MARGIN_BIN_DIFFERENCE:
            continue
        delta = (
            correct_z.iloc[correct_index].to_numpy(dtype=float)
            - error_z.iloc[error_index].to_numpy(dtype=float)
        )
        distance = float(np.linalg.norm(delta) / np.sqrt(len(PHYSICAL_FEATURES)))
        if distance > MAXIMUM_STANDARDIZED_PAIR_DISTANCE:
            continue
        edge_rows.append({
            'edge_index': len(edge_rows),
            'correct_index': int(correct_index), 'error_index': int(error_index),
            'correct_pair_id': str(correct['pair_id']),
            'error_pair_id': str(error['pair_id']),
            'diagnostic_cell': str(correct['diagnostic_cell']),
            'margin_bin_difference': int(margin_difference),
            'standardized_pair_distance': distance,
            'tie_break': int(stable_key((
                'edge', correct['pair_id'], error['pair_id']
            ))[:8], 16) / 2**32,
        })
edge_frame = pd.DataFrame(edge_rows)
assert len(edge_frame) > 0
edge_frame.to_csv(EDGE_TABLE_PATH, index=False)
correct_degree = edge_frame.groupby('correct_index').size()
print('Admissible edges:', len(edge_frame))
print('Correct pairs with at least one edge:', len(correct_degree), '/', len(correct_frame))
print('Edge distance range:',
      edge_frame['standardized_pair_distance'].min(),
      edge_frame['standardized_pair_distance'].max())


## 2. Discrete common-support optimization


In [ ]:
%%cell_progress 6/10 Search for the largest physically balanced subset
cell_stage('Solving precommitted binary matching programs')

def actual_balance(selected_edges):
    correct_ids = selected_edges['correct_pair_id'].tolist()
    error_ids = selected_edges['error_pair_id'].tolist()
    correct = correct_frame.set_index('pair_id').loc[correct_ids]
    error = error_frame.set_index('pair_id').loc[error_ids]
    rows = []
    for feature in PHYSICAL_FEATURES:
        left = correct[feature].to_numpy(dtype=float)
        right = error[feature].to_numpy(dtype=float)
        pooled = np.sqrt((left.var(ddof=1) + right.var(ddof=1)) / 2)
        difference = float(left.mean() - right.mean())
        smd = difference / pooled if pooled > 0 else (0.0 if difference == 0 else np.inf)
        rows.append({
            'feature': feature,
            'both_correct_mean': float(left.mean()),
            'right_error_mean': float(right.mean()),
            'standardized_mean_difference': float(smd),
            'absolute_smd': abs(float(smd)),
        })
    return pd.DataFrame(rows)

def solve_matching(balance_bound):
    edge_count = len(edge_frame)
    matrix_rows, lower, upper = [], [], []
    for correct_index in range(len(correct_frame)):
        row = np.zeros(edge_count)
        row[edge_frame.index[edge_frame['correct_index'].eq(correct_index)]] = 1.0
        matrix_rows.append(row); lower.append(-np.inf); upper.append(1.0)
    for error_index in range(len(error_frame)):
        row = np.zeros(edge_count)
        row[edge_frame.index[edge_frame['error_index'].eq(error_index)]] = 1.0
        matrix_rows.append(row); lower.append(-np.inf); upper.append(1.0)
    for feature in PHYSICAL_FEATURES:
        differences = np.asarray([
            float(correct_z.loc[int(edge['correct_index']), feature]
                  - error_z.loc[int(edge['error_index']), feature])
            for _, edge in edge_frame.iterrows()
        ])
        matrix_rows.append(differences - balance_bound)
        lower.append(-np.inf); upper.append(0.0)
        matrix_rows.append(-differences - balance_bound)
        lower.append(-np.inf); upper.append(0.0)
    constraint = LinearConstraint(
        csr_matrix(np.vstack(matrix_rows)),
        np.asarray(lower), np.asarray(upper),
    )
    objective = np.asarray([
        -100.0 + float(row['standardized_pair_distance'])
        + 1e-4 * float(row['tie_break'])
        for _, row in edge_frame.iterrows()
    ])
    result = milp(
        c=objective, integrality=np.ones(edge_count),
        bounds=Bounds(np.zeros(edge_count), np.ones(edge_count)),
        constraints=constraint,
        options={'time_limit': 120.0, 'mip_rel_gap': 0.0},
    )
    assert result.success, (balance_bound, result.message)
    selected = edge_frame[np.asarray(result.x) > 0.5].copy()
    balance = actual_balance(selected) if len(selected) >= 2 else pd.DataFrame()
    maximum = float(balance['absolute_smd'].max()) if len(balance) else np.inf
    return selected, balance, {
        'balance_bound': float(balance_bound),
        'matched_units': int(len(selected)),
        'maximum_absolute_smd': maximum,
        'mean_pair_distance': float(
            selected['standardized_pair_distance'].mean()
        ) if len(selected) else np.nan,
        'maximum_pair_distance': float(
            selected['standardized_pair_distance'].max()
        ) if len(selected) else np.nan,
        'post_match_balance_passed': bool(maximum <= MAXIMUM_ABSOLUTE_SMD),
        'minimum_size_passed': bool(len(selected) >= MINIMUM_MATCHED_UNITS),
    }

solutions = []
search_rows = []
for bound in tqdm(BALANCE_BOUNDS, desc='MILP balance path', unit='bound'):
    selected, balance, summary = solve_matching(bound)
    solutions.append((selected, balance, summary))
    search_rows.append(summary)
search_frame = pd.DataFrame(search_rows)
search_frame.to_csv(SEARCH_PATH, index=False)
display(search_frame.round(4))

eligible = [
    item for item in solutions
    if item[2]['post_match_balance_passed']
    and item[2]['minimum_size_passed']
]
if eligible:
    eligible.sort(key=lambda item: (
        -item[2]['matched_units'],
        item[2]['maximum_absolute_smd'],
        item[2]['mean_pair_distance'],
    ))
    selected_edges, selected_balance, selected_summary = eligible[0]
else:
    solutions.sort(key=lambda item: (
        -int(item[2]['post_match_balance_passed']),
        -item[2]['matched_units'], item[2]['maximum_absolute_smd'],
    ))
    selected_edges, selected_balance, selected_summary = solutions[0]
print('Selected optimization:', selected_summary)


In [ ]:
%%cell_progress 7/10 Audit the selected common-support cohort
cell_stage('Checking uniqueness, calipers, balance, and behavior-only provenance')

selected_edges = selected_edges.sort_values([
    'diagnostic_cell', 'standardized_pair_distance', 'correct_pair_id'
]).reset_index(drop=True)
selected_matches = []
for index, row in selected_edges.iterrows():
    selected_matches.append({
        'match_id': f'common_support_{index:02d}',
        'both_correct_pair_id': str(row['correct_pair_id']),
        'right_error_pair_id': str(row['error_pair_id']),
        'diagnostic_cell': str(row['diagnostic_cell']),
        'margin_bin_difference': int(row['margin_bin_difference']),
        'standardized_pair_distance': float(row['standardized_pair_distance']),
    })
write_jsonl_atomic(SELECTED_MATCH_PATH, selected_matches)
selected_balance.to_csv(BALANCE_PATH, index=False)

assert len({row['both_correct_pair_id'] for row in selected_matches}) == len(selected_matches)
assert len({row['right_error_pair_id'] for row in selected_matches}) == len(selected_matches)
assert all(
    row['margin_bin_difference'] <= MAXIMUM_MARGIN_BIN_DIFFERENCE
    for row in selected_matches
)
assert all(
    row['standardized_pair_distance'] <= MAXIMUM_STANDARDIZED_PAIR_DISTANCE
    for row in selected_matches
)

match_by_pair = {}
for row in selected_matches:
    match_by_pair[row['both_correct_pair_id']] = (row['match_id'], 'both_correct')
    match_by_pair[row['right_error_pair_id']] = (row['match_id'], 'right_error')
selected_scenes = []
for pair_id, (match_id, cohort) in match_by_pair.items():
    for candidate in candidate_by_pair[pair_id]:
        side = str(candidate['exact_semantic'])
        baseline_status = 'error' if cohort == 'right_error' and side == 'right' else 'correct'
        selected_scenes.append({
            **candidate, 'match_id': match_id, 'cohort': cohort,
            'side': side, 'baseline_status': baseline_status,
        })
write_jsonl_atomic(SELECTED_SCENE_PATH, selected_scenes)
assert len(selected_scenes) == 4 * len(selected_matches)
assert Counter((row['cohort'], row['side']) for row in selected_scenes) == {
    (cohort, side): len(selected_matches)
    for cohort in ('both_correct', 'right_error')
    for side in ('left', 'right')
}

MATCHED_UNITS = len(selected_matches)
MAXIMUM_SELECTED_SMD = float(selected_balance['absolute_smd'].max())
READINESS_GATE = bool(
    MATCHED_UNITS >= MINIMUM_MATCHED_UNITS
    and MAXIMUM_SELECTED_SMD <= MAXIMUM_ABSOLUTE_SMD
)
PREFERRED_TARGET_GATE = bool(
    READINESS_GATE and MATCHED_UNITS >= TARGET_MATCHED_UNITS
)
if PREFERRED_TARGET_GATE:
    PRIMARY_DECISION = 'common_support_ready_at_preferred_size'
elif READINESS_GATE:
    PRIMARY_DECISION = 'common_support_ready_at_minimum_size'
else:
    PRIMARY_DECISION = 'dataset_v1_common_support_is_insufficient'

decision = {
    'primary_decision': PRIMARY_DECISION,
    'matched_units': MATCHED_UNITS,
    'maximum_absolute_smd': MAXIMUM_SELECTED_SMD,
    'readiness_gate': READINESS_GATE,
    'preferred_target_gate': PREFERRED_TARGET_GATE,
    'selected_optimization': selected_summary,
    'uses_hidden_states': False,
    'uses_intervention_results': False,
    'claim_boundary': (
        'Readiness concerns overlap within the behavior-defined Dataset v1 '
        'cohorts. It does not establish an error-specific mechanism.'
    ),
}
write_json_atomic(ANALYSIS_DIR / 'common_support_decision.json', decision)
display(selected_balance.sort_values('absolute_smd', ascending=False).round(4))
print('Primary decision:', PRIMARY_DECISION)


In [ ]:
%%cell_progress 8/10 Diagnose overlap limits and targeted data needs
cell_stage('Locating the physical dimensions that limit balanced comparison')

range_rows = []
for feature in PHYSICAL_FEATURES:
    correct_values = correct_frame[feature].to_numpy(dtype=float)
    error_values = error_frame[feature].to_numpy(dtype=float)
    overlap_low = max(correct_values.min(), error_values.min())
    overlap_high = min(correct_values.max(), error_values.max())
    pooled_low = min(correct_values.min(), error_values.min())
    pooled_high = max(correct_values.max(), error_values.max())
    pooled_width = pooled_high - pooled_low
    overlap_fraction = (
        max(0.0, overlap_high - overlap_low) / pooled_width
        if pooled_width > 0 else 1.0
    )
    balance_row = selected_balance.set_index('feature').loc[feature]
    range_rows.append({
        'feature': feature,
        'both_correct_min': float(correct_values.min()),
        'both_correct_max': float(correct_values.max()),
        'right_error_min': float(error_values.min()),
        'right_error_max': float(error_values.max()),
        'range_overlap_fraction': float(overlap_fraction),
        'selected_absolute_smd': float(balance_row['absolute_smd']),
    })
overlap_frame = pd.DataFrame(range_rows).sort_values(
    ['selected_absolute_smd', 'range_overlap_fraction'], ascending=[False, True]
)
overlap_frame.to_csv(ANALYSIS_DIR / 'overlap_diagnostics.csv', index=False)

needs = overlap_frame.head(5).copy()
needs['recommendation'] = needs.apply(
    lambda row: (
        'Add independently rendered scenes in the shared range and screen '
        'for both-correct behavior; do not merely duplicate the dominant '
        f"right-error region for {row['feature']}."
    ), axis=1,
)
needs.to_csv(ANALYSIS_DIR / 'targeted_data_needs.csv', index=False)
display(overlap_frame.round(4))


## 3. Readiness decision and targeted data guidance


In [ ]:
%%cell_progress 9/10 Save the common-support figure
cell_stage('Rendering optimization, balance, and overlap panels')

fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
axes[0].plot(
    search_frame['balance_bound'], search_frame['matched_units'],
    marker='o', color='#2563EB', linewidth=2,
)
axes[0].axhline(MINIMUM_MATCHED_UNITS, color='#DC2626', linestyle='--',
                label='Minimum size')
axes[0].axhline(TARGET_MATCHED_UNITS, color='#059669', linestyle=':',
                label='Preferred size')
axes[0].invert_xaxis()
axes[0].set_xlabel('Optimization mean-balance bound')
axes[0].set_ylabel('Matched units')
axes[0].set_title('A. Size along the balance path')
axes[0].legend(frameon=False)

balance_plot = selected_balance.sort_values('absolute_smd')
colors = ['#DC2626' if value > MAXIMUM_ABSOLUTE_SMD else '#0F766E'
          for value in balance_plot['absolute_smd']]
axes[1].barh(
    balance_plot['feature'], balance_plot['standardized_mean_difference'],
    color=colors,
)
axes[1].axvline(MAXIMUM_ABSOLUTE_SMD, color='black', linestyle='--')
axes[1].axvline(-MAXIMUM_ABSOLUTE_SMD, color='black', linestyle='--')
axes[1].set_xlabel('Selected standardized mean difference')
axes[1].set_title('B. Post-match physical balance')

overlap_plot = overlap_frame.sort_values('range_overlap_fraction')
axes[2].barh(
    overlap_plot['feature'], overlap_plot['range_overlap_fraction'],
    color='#7C3AED',
)
axes[2].set_xlim(0, 1.05)
axes[2].set_xlabel('Shared fraction of observed range')
axes[2].set_title('C. Full-cohort range overlap')
fig.suptitle(PRIMARY_DECISION, y=1.03)
fig.tight_layout()
figure_path = FIGURE_DIR / 'rightside_common_support_readiness.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)


In [ ]:
%%cell_progress 10/10 Write the readiness report
cell_stage('Saving the decision card and completion manifest')

worst = selected_balance.sort_values('absolute_smd', ascending=False).head(5)
lines = [
    '# Right-Side Common-Support Matching Readiness', '',
    f'**Primary decision:** `{PRIMARY_DECISION}`', '',
    '## Result', '',
    f'- Selected matched units: {MATCHED_UNITS}',
    f'- Maximum absolute SMD: {MAXIMUM_SELECTED_SMD:.4f}',
    f'- Minimum-size gate: `{READINESS_GATE}`',
    f'- Preferred-size gate: `{PREFERRED_TARGET_GATE}`', '',
    '## Worst remaining physical imbalances', '',
]
for _, row in worst.iterrows():
    lines.append(
        f"- {row['feature']}: SMD={row['standardized_mean_difference']:.4f}"
    )
lines.extend([
    '', '## Interpretation', '',
    (
        'A passing result licenses a smaller matched causal confirmation using '
        'the saved scene manifest. A failure means the existing behavior '
        'cohorts occupy different physical regions; causal effects from the '
        'previous forced 14-pair match must remain suggestive.'
    ), '', '## Provenance', '',
    '- Hidden states read: no',
    '- Intervention outputs read: no',
    '- Causal effect sizes used for selection: no', '',
])
report_path = ANALYSIS_DIR / 'RESULT_CARD.md'
report_path.write_text('\n'.join(lines), encoding='utf-8')
run_complete = {
    'status': 'complete',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'primary_decision': PRIMARY_DECISION,
    'matched_units': MATCHED_UNITS,
    'maximum_absolute_smd': MAXIMUM_SELECTED_SMD,
    'readiness_gate': READINESS_GATE,
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'selected_match_sha256': sha256_file(SELECTED_MATCH_PATH),
    'selected_scene_sha256': sha256_file(SELECTED_SCENE_PATH),
    'decision_sha256': sha256_file(
        ANALYSIS_DIR / 'common_support_decision.json'
    ),
    'figure': str(figure_path), 'report': str(report_path),
}
write_json_atomic(OUTPUT_ROOT / 'RUN_COMPLETE.json', run_complete)
print('Saved:', report_path)
print('RUN_COMPLETE:', OUTPUT_ROOT / 'RUN_COMPLETE.json')
